# Day 37 — State-compression DP and Held-Karp for the TSP

A set of at most about twenty things fits in one machine integer, and once a set
is an integer it can be a DP index. That single change of representation turns
"try every ordering" — `(n-1)!` of them — into "fill a table of `2^n · n` cells".

This notebook builds the idea up from bit tricks, solves the travelling salesman
problem exactly on eight stops, and then shows the two ways it silently goes
wrong: dropping `last` from the state, and being greedy instead.

## 1. A set is an integer

Bit `i` of the mask answers "is element `i` in the set?". All the set operations
become one arithmetic expression each, and the whole set fits in a register.

In [1]:
import itertools, math, time
from functools import lru_cache
INF = float('inf')

def has(mask, i):
    """Is element i in the set?"""
    return (mask >> i) & 1 == 1


def add(mask, i):
    """The set with i added (masks are immutable ints - this returns a new one)."""
    return mask | (1 << i)


def remove(mask, i):
    """The set with i removed."""
    return mask & ~(1 << i)


def members(mask):
    """The elements of the set, ascending."""
    i = 0
    while mask:
        if mask & 1:
            yield i
        mask >>= 1
        i += 1


def popcount(mask):
    """How many elements.  int.bit_count() exists from Python 3.10."""
    return bin(mask).count('1')

In [2]:
m = 0
for i in (0, 2, 3):
    m = add(m, i)
print(f'add 0,2,3 -> {m} = 0b{m:08b}  members {list(members(m))}')
print(f'remove 2  -> {remove(m, 2)} = 0b{remove(m, 2):08b}')
print(f'popcount  -> {popcount(m)}')

add 0,2,3 -> 13 = 0b00001101  members [0, 2, 3]
remove 2  -> 9 = 0b00001001
popcount  -> 3


## 2. Enumerating subsets, and why the loop has to be a do-while

`sub = (sub - 1) & mask` walks the subsets of `mask` downwards: subtracting one
borrows through the low zeros, and the `&` puts back only the bits that belong
to the mask. Writing it as a plain `while sub:` loop silently drops the empty
set, which is a perfectly legal subset and usually the base case.

In [3]:
def subsets_of(mask):
    """Every subset of mask, including mask itself and the empty set.

    The trick is `sub = (sub - 1) & mask`: subtracting one borrows through the
    low zeros, and the & puts back only the bits that belong to mask.  It has
    to be a do-while, because the empty set is a legal subset and the loop
    condition would skip it.
    """
    sub = mask
    while True:
        yield sub
        if sub == 0:
            break
        sub = (sub - 1) & mask


def subsets_of_buggy(mask):
    """The same loop written as a plain `while sub:` - drops the empty set."""
    sub = mask
    while sub:
        yield sub
        sub = (sub - 1) & mask


def subset_pair_count(n):
    """sum over masks of 2^popcount(mask) - the cost of 'for every subset of

    every mask'.  Each element is in sub, in mask-but-not-sub, or outside:
    three choices, so the total is exactly 3^n, not 4^n.
    """
    return sum(1 << popcount(m) for m in range(1 << n))

In [4]:
print([format(s, '04b') for s in subsets_of(0b1011)])
print([format(s, '04b') for s in subsets_of_buggy(0b1011)], '<- empty set missing')
print()
print(f'every subset of every mask, n=10: {subset_pair_count(10):,} = 3^10 = {3**10:,}')
print(f'not 4^10 = {4**10:,}: each element is in sub, in mask but not sub, or outside')

['1011', '1010', '1001', '1000', '0011', '0010', '0001', '0000']
['1011', '1010', '1001', '1000', '0011', '0010', '0001'] <- empty set missing

every subset of every mask, n=10: 59,049 = 3^10 = 59,049
not 4^10 = 1,048,576: each element is in sub, in mask but not sub, or outside


## 3. The fill order is free

Day 36 spent its whole budget on getting the fill order of an interval DP right.
Here there is nothing to get right: every transition *adds* elements, and adding
a bit can only make the integer larger, so plain `for mask in range(1 << n)` is
already a topological order of the subset lattice.

In [5]:
def fill_order_is_free(n=4):
    """Why `for mask in range(1 << n)` needs no sorting.

    Every transition of a subset DP adds elements, and adding a bit can only
    make the integer larger.  So numeric order on the masks is already a
    topological order on the subset lattice - the thing that had to be worked
    out by hand for interval DP comes for free here.
    """
    bad = [(mask, sub) for mask in range(1 << n)
           for sub in subsets_of(mask) if sub > mask]
    return len(bad)

In [6]:
print('subsets numerically larger than their superset:', fill_order_is_free(8))

subsets numerically larger than their superset: 0


## 4. The instance: eight stops, one closed tour

Seven of these sit on the ring road around Taiwan; Sun Moon Lake sits inland,
and that one inland stop is what makes the problem interesting later on.

In [7]:
CITIES = [('Taipei', 157.6, 336.3), ('Hsinchu', 98.0, 312.0), ('Taichung', 68.7, 238.7), ('Sun Moon Lake', 90.9, 203.1), ('Tainan', 21.2, 109.9), ('Kaohsiung', 30.3, 69.9), ('Taitung', 115.1, 84.4), ('Hualien', 162.6, 220.9)]
SHORT = ['TPE', 'HSZ', 'TXG', 'SML', 'TNN', 'KHH', 'TTT', 'HUN']

def distances(cities=CITIES):
    n = len(cities)
    return [[round(math.hypot(cities[i][1] - cities[j][1],
                              cities[i][2] - cities[j][2]))
             for j in range(n)] for i in range(n)]

DIST = distances()
N = len(DIST)

def tour_cost(D, tour):
    return sum(D[tour[k]][tour[(k + 1) % len(tour)]] for k in range(len(tour)))

print('      ' + ' '.join(f'{s:>5}' for s in SHORT))
for i, row in enumerate(DIST):
    print(f'{SHORT[i]:>5} ' + ' '.join(f'{v:5d}' for v in row))

        TPE   HSZ   TXG   SML   TNN   KHH   TTT   HUN
  TPE     0    64   132   149   264   295   255   116
  HSZ    64     0    79   109   216   251   228   112
  TXG   132    79     0    42   137   173   161    96
  SML   149   109    42     0   116   146   121    74
  TNN   264   216   137   116     0    41    97   180
  KHH   295   251   173   146    41     0    86   201
  TTT   255   228   161   121    97    86     0   145
  HUN   116   112    96    74   180   201   145     0


## 5. Brute force, which is right and hopeless

City 0 can always be called the start, because a tour is a cycle with no
beginning. That leaves `(n-1)!` orderings — 5,040 here, and 87 billion at
fifteen stops.

In [8]:
def tsp_brute(D):
    """Try every ordering of cities 1..n-1.  (n-1)! of them."""
    n = len(D)
    best, best_tour, tried = INF, None, 0
    for perm in itertools.permutations(range(1, n)):
        tried += 1
        cost = tour_cost(D, (0,) + perm)
        if cost < best:
            best, best_tour = cost, [0] + list(perm)
    return best, best_tour, tried

In [9]:
t0 = time.perf_counter()
best, tour, tried = tsp_brute(DIST)
ms = (time.perf_counter() - t0) * 1000
print(f'{best} km after {tried} permutations, {ms:.1f} ms')
print(' -> '.join(SHORT[i] for i in tour))

689 km after 5040 permutations, 8.2 ms
TPE -> HSZ -> TXG -> SML -> TNN -> KHH -> TTT -> HUN


## 6. Held-Karp: the state is (set visited, where I am standing)

`dp[mask][last]` is the cheapest way to have started at city 0, visited exactly
the cities in `mask`, and be standing on `last` right now.

Both halves of that state earn their keep. The set says what has already been
spent; `last` says what the next edge is going to cost. Two different routes
that have seen the same four cities are interchangeable from here on **only if
they end in the same place** — that is the whole compression, and it is why the
table has `2^n · n` cells instead of `n!` rows.

In [10]:
def held_karp(D):
    """Exact TSP in O(2^n * n^2) time and O(2^n * n) space.

    dp[mask][last] = cheapest way to start at city 0, visit exactly the cities
    in mask, and be standing on `last` right now.  Both halves matter: the set
    says what is already spent, `last` says what the next edge will cost.
    """
    n = len(D)
    size = 1 << n
    dp = [[INF] * n for _ in range(size)]
    parent = [[-1] * n for _ in range(size)]
    dp[1][0] = 0                                   # start at city 0, nothing else seen

    # Plain increasing integer order is already a legal fill order: every
    # subset of a mask is numerically smaller than the mask.
    for mask in range(size):
        if not has(mask, 0):
            continue
        for last in range(n):
            cur = dp[mask][last]
            if cur == INF or not has(mask, last):
                continue
            for nxt in range(n):
                if has(mask, nxt):
                    continue
                nmask = add(mask, nxt)
                cand = cur + D[last][nxt]
                if cand < dp[nmask][nxt]:
                    dp[nmask][nxt] = cand
                    parent[nmask][nxt] = last

    full = size - 1
    best, end = INF, -1
    for last in range(1, n):                       # close the loop back to 0
        cand = dp[full][last] + D[last][0]
        if cand < best:
            best, end = cand, last

    tour, mask, cur = [], full, end                # walk the parent pointers back
    while cur != -1:
        tour.append(cur)
        prev = parent[mask][cur]
        mask = remove(mask, cur)
        cur = prev
    tour.reverse()
    return best, tour, dp

In [11]:
t0 = time.perf_counter()
cost, tour, dp = held_karp(DIST)
ms = (time.perf_counter() - t0) * 1000
print(f'{cost} km, {ms:.1f} ms')
print(' -> '.join(SHORT[i] for i in tour))
print()
print('same answer as brute force:', cost == best)

689 km, 1.3 ms
TPE -> HUN -> TTT -> KHH -> TNN -> SML -> TXG -> HSZ

same answer as brute force: True


Top-down with `lru_cache` is the same recurrence; the cache size shows how many
states the memo actually touched.

In [12]:
def held_karp_memo(D):
    """The same recurrence written top-down, for the people who prefer it."""
    n = len(D)
    full = (1 << n) - 1

    @lru_cache(maxsize=None)
    def go(mask, last):
        if mask == full:
            return D[last][0]
        best = INF
        for nxt in range(n):
            if not has(mask, nxt):
                best = min(best, D[last][nxt] + go(add(mask, nxt), nxt))
        return best

    ans = go(1, 0)
    return ans, go.cache_info().currsize

In [13]:
ans, states = held_karp_memo(DIST)
print(f'{ans} km from {states} memoised states (the full table would be {(1<<N)*N} cells)')

689 km from 449 memoised states (the full table would be 2048 cells)


## 7. The silent failure: dropping `last`

Suppose the state is just `dp[mask]`. Nothing crashes — you can still write a
recurrence, because there is still *something* sensible to pay when a new city
joins the set: the distance to whichever visited city is nearest.

That recurrence is not broken. It just answers a different question. It grows a
**tree**, and a tree is not a tour: nobody ever comes home. The number it
returns is the minimum spanning tree weight, which is a classic *lower bound* on
the tour — so it is always smaller than the true optimum, always plausible, and
never flagged.

In [14]:
def tsp_mask_only(D):
    """dp[mask] alone - the version that returns a number and not a tour.

    Without `last` there is no way to price the next edge, so the only thing
    that can be written is "connect the new city to whichever visited city is
    nearest".  That is a legal recurrence.  It just answers a different
    question: it builds a *tree*, not a tour.
    """
    n = len(D)
    dp = [INF] * (1 << n)
    dp[1] = 0
    for mask in range(1 << n):
        if dp[mask] == INF:
            continue
        for nxt in range(n):
            if has(mask, nxt):
                continue
            hop = min(D[i][nxt] for i in members(mask))
            nmask = add(mask, nxt)
            if dp[mask] + hop < dp[nmask]:
                dp[nmask] = dp[mask] + hop
    return dp[(1 << n) - 1]


def mst_weight(D):
    """Prim's algorithm, for comparison with tsp_mask_only."""
    n = len(D)
    seen = {0}
    total = 0
    while len(seen) < n:
        best = min(((D[i][j], j) for i in seen for j in range(n) if j not in seen))
        total += best[0]
        seen.add(best[1])
    return total

In [15]:
wrong = tsp_mask_only(DIST)
print(f'dp[mask] only          {wrong} km')
print(f'minimum spanning tree  {mst_weight(DIST)} km   <- the same number')
print(f'true optimum           {cost} km')
assert wrong == mst_weight(DIST) < cost

dp[mask] only          502 km
minimum spanning tree  502 km   <- the same number
true optimum           689 km


## 8. The other shortcut: nearest neighbour

Greedy drives to the closest unvisited stop. On this map it goes inland to Sun
Moon Lake and then hops to Hualien — the nearest place — which strands the
entire south, and the trip home is enormous. The result is a real, valid,
plausible-looking itinerary that happens to be 15% too long.

In [16]:
def nearest_neighbour(D, start=0):
    """Always drive to the closest place you have not been to yet."""
    n = len(D)
    cur, seen, tour, cost = start, {start}, [start], 0
    while len(seen) < n:
        nxt = min((j for j in range(n) if j not in seen), key=lambda j: (D[cur][j], j))
        cost += D[cur][nxt]
        cur = nxt
        seen.add(nxt)
        tour.append(nxt)
    return cost + D[cur][start], tour

In [17]:
ncost, ntour = nearest_neighbour(DIST)
print(f'greedy {ncost} km  ' + ' -> '.join(SHORT[i] for i in ntour))
print(f'exact  {cost} km  ' + ' -> '.join(SHORT[i] for i in tour))
print(f'{ncost - cost} km worse, {100 * (ncost - cost) / cost:.1f}% over')

greedy 795 km  TPE -> HSZ -> TXG -> SML -> HUN -> TTT -> KHH -> TNN
exact  689 km  TPE -> HUN -> TTT -> KHH -> TNN -> SML -> TXG -> HSZ
106 km worse, 15.4% over


## 9. Two exponentials are not the same exponential

`2^n · n^2` is still exponential — Held-Karp does not make the TSP tractable,
it moves the wall. But it moves it a long way: the ratio to `(n-1)!` is already
346x at thirteen cities.

In [18]:
def ring(n, seed=7):
    """A deterministic n-city instance for timing, no random module needed."""
    cs = []
    for i in range(n):
        ang = 2 * math.pi * ((i * seed) % n) / n
        r = 100 + 37 * ((i * i * seed) % 11)
        cs.append((f'c{i}', r * math.cos(ang), r * math.sin(ang)))
    return distances(cs)


def crossover(brute_upto=10, hk_upto=15):
    """Measured milliseconds for both exact methods as n grows."""
    rows = []
    for n in range(8, hk_upto + 1):
        D = ring(n)
        _, hms = timed(lambda d: held_karp(d), D)
        bms = None
        if n <= brute_upto:
            _, bms = timed(lambda d: tsp_brute(d), D)
        rows.append((n, bms, hms))
    return rows


def timed(fn, D):
    t0 = time.perf_counter()
    out = fn(D)
    return out, (time.perf_counter() - t0) * 1000


def work_table(sizes=(5, 8, 10, 13, 15, 20)):
    """(n, (n-1)! permutations, 2^n * n^2 DP cells-times-transitions)."""
    return [(n, math.factorial(n - 1), (1 << n) * n * n) for n in sizes]

In [19]:
print('   n                   (n-1)!       2^n * n^2           ratio')
for n, f, d in work_table():
    print(f'{n:4d}  {f:24,d}  {d:14,d}  {f/d:14.1f}x')
print()
print('measured, milliseconds')
print('   n    brute force    Held-Karp')
for n, bms, hms in crossover(brute_upto=10, hk_upto=14):
    bs = f'{bms:11.1f}' if bms is not None else '          -'
    print(f'{n:4d}  {bs}  {hms:11.1f}')

   n                   (n-1)!       2^n * n^2           ratio
   5                        24             800             0.0x
   8                     5,040          16,384             0.3x
  10                   362,880         102,400             3.5x
  13               479,001,600       1,384,448           346.0x
  15            87,178,291,200       7,372,800         11824.3x
  20   121,645,100,408,832,000     419,430,400     290024519.9x

measured, milliseconds
   n    brute force    Held-Karp
   8          7.4          1.2
   9         66.3          4.6
  10        558.6          7.5
  11            -         14.2
  12            -         33.7
  13            -         77.0
  14            -        196.1


## 10. LeetCode 847 — Shortest Path Visiting All Nodes

**The task.** Given a connected undirected graph, find the length of the
shortest walk that visits every node at least once. The walk may start at any
node, may end at any node, and may pass through nodes and edges as many times as
it likes.

**Input / output.** `graph` is an adjacency list: `graph[i]` lists the
neighbours of node `i`. The function returns one integer — the number of edges
in the shortest such walk.

**Example.** `graph = [[1,2,3],[0],[0],[0]]` is a star with node 0 at the
centre. The answer is 4: `1 → 0 → 2 → 0 → 3` uses four edges, and the centre has
to be crossed twice because the leaves are not connected to one another.

**Constraints.** `n ≤ 12`, which is the tell: `2^12 · 12` states is nothing,
whereas anything factorial in 12 is not. Revisits are allowed, so this is not a
Hamiltonian path — and because every edge costs the same 1, it is BFS rather
than DP, but over exactly the same `(mask, node)` state space. The state already
records a revisit as "same mask, different node", so allowing them costs
nothing.

[leetcode.com/problems/shortest-path-visiting-all-nodes](https://leetcode.com/problems/shortest-path-visiting-all-nodes/)

In [20]:
def shortest_path_length(graph):
    """Fewest edges in a walk that touches every node.  Revisits allowed.

    Because every edge costs the same 1, this is BFS, not DP - but over the
    same (mask, node) state space.  Revisits are free to allow: the state
    already records them as 'same mask, different node'.
    """
    from collections import deque
    n = len(graph)
    full = (1 << n) - 1
    if n == 1:
        return 0
    seen = {(1 << i, i) for i in range(n)}
    q = deque(((1 << i, i, 0) for i in range(n)))
    while q:
        mask, node, d = q.popleft()
        for nb in graph[node]:
            nmask = mask | (1 << nb)
            if nmask == full:
                return d + 1
            if (nmask, nb) not in seen:
                seen.add((nmask, nb))
                q.append((nmask, nb, d + 1))
    return 0

In [21]:
print(shortest_path_length([[1,2,3],[0],[0],[0]]))
print(shortest_path_length([[1],[0,2,4],[1,3,4],[2],[1,2]]))

4
4


## 11. LeetCode 943 — Find the Shortest Superstring

**The task.** Given a list of strings, produce the shortest possible string that
contains every one of them as a substring. Any shortest answer is accepted.

**Input / output.** `words` is a list of strings, no one of which is a substring
of another. The function returns one string.

**Example.** For `["catg","ctaagt","gcta","ttca","atgcatc"]` the characters add
up to 25, but chaining them in the order `gcta, ctaagt, ttca, catg, atgcatc` —
each pair sharing a suffix with the next one's prefix — saves 9 characters and
gives the 16-character `gctaagttcatgcatc`.

**Constraints.** At most 12 words, each at most 20 characters. Twelve is the
bitmask signal again. Maximising total overlap is the same DP as minimising
total distance, so this is a travelling salesman in disguise — but an **open
path**, not a closed tour, and that difference is a trap: in a tour any city may
be called the start, so seeding only `dp[1][0]` is free; here the first word is
a real decision, and seeding one start still returns a valid superstring, merely
a longer one.

[leetcode.com/problems/find-the-shortest-superstring](https://leetcode.com/problems/find-the-shortest-superstring/)

In [22]:
def overlap(a, b):
    """Length of the longest suffix of a that is also a prefix of b."""
    for k in range(min(len(a), len(b)), 0, -1):
        if a.endswith(b[:k]):
            return k
    return 0


def shortest_superstring(words):
    """Shortest string containing every word - an open TSP in disguise.

    Maximise total overlap instead of minimising distance; the state is the
    same (set of words used, which word is last).
    """
    n = len(words)
    ov = [[overlap(words[i], words[j]) if i != j else 0 for j in range(n)]
          for i in range(n)]
    dp = [[-1] * n for _ in range(1 << n)]
    parent = [[-1] * n for _ in range(1 << n)]
    for i in range(n):
        dp[1 << i][i] = 0
    for mask in range(1 << n):
        for last in range(n):
            if dp[mask][last] < 0 or not has(mask, last):
                continue
            for nxt in range(n):
                if has(mask, nxt):
                    continue
                nmask = add(mask, nxt)
                cand = dp[mask][last] + ov[last][nxt]
                if cand > dp[nmask][nxt]:
                    dp[nmask][nxt] = cand
                    parent[nmask][nxt] = last
    full = (1 << n) - 1
    end = max(range(n), key=lambda i: dp[full][i])
    order, mask, cur = [], full, end
    while cur != -1:
        order.append(cur)
        prev = parent[mask][cur]
        mask = remove(mask, cur)
        cur = prev
    order.reverse()
    out = words[order[0]]
    for k in range(1, len(order)):
        out += words[order[k]][ov[order[k - 1]][order[k]]:]
    return out, order, dp[full][end]


def shortest_superstring_fixed_start(words):
    """The habit carried over from TSP, applied where it does not hold.

    In a closed tour any city can be called the start, so seeding only
    dp[1][0] is free.  A superstring is an open path: the first word is a real
    decision.  Seeding one start still produces a valid superstring - just a
    longer one - so nothing complains.
    """
    n = len(words)
    ov = [[overlap(words[i], words[j]) if i != j else 0 for j in range(n)]
          for i in range(n)]
    dp = [[-1] * n for _ in range(1 << n)]
    parent = [[-1] * n for _ in range(1 << n)]
    dp[1][0] = 0                                  # <- only word 0 may go first
    for mask in range(1 << n):
        for last in range(n):
            if dp[mask][last] < 0 or not has(mask, last):
                continue
            for nxt in range(n):
                if has(mask, nxt):
                    continue
                nmask = add(mask, nxt)
                cand = dp[mask][last] + ov[last][nxt]
                if cand > dp[nmask][nxt]:
                    dp[nmask][nxt] = cand
                    parent[nmask][nxt] = last
    full = (1 << n) - 1
    end = max(range(n), key=lambda i: dp[full][i])
    order, mask, cur = [], full, end
    while cur != -1:
        order.append(cur)
        prev = parent[mask][cur]
        mask = remove(mask, cur)
        cur = prev
    order.reverse()
    out = words[order[0]]
    for k in range(1, len(order)):
        out += words[order[k]][ov[order[k - 1]][order[k]]:]
    return out, order

In [23]:
words = ['catg', 'ctaagt', 'gcta', 'ttca', 'atgcatc']
sup, order, gained = shortest_superstring(words)
print('order  ', [words[i] for i in order])
print(f'{sum(len(w) for w in words)} characters -> {len(sup)}, overlap saved {gained}')
print('answer ', sup)
print()
bad, bad_order = shortest_superstring_fixed_start(words)
print('fixing the first word:', bad, f'({len(bad)} characters, still valid)')

order   ['gcta', 'ctaagt', 'ttca', 'catg', 'atgcatc']
25 characters -> 16, overlap saved 9
answer  gctaagttcatgcatc

fixing the first word: catgcatcgctaagttca (18 characters, still valid)


## 12. LeetCode 1125 — Smallest Sufficient Team

**The task.** A project needs a set of skills. Each candidate has some subset of
those skills. Form the smallest team that between them covers every required
skill.

**Input / output.** `req_skills` is a list of skill names and `people[i]` is the
list of skills person `i` has. The function returns a list of person indices —
any smallest team is accepted.

**Example.** With `req_skills = ["java","nodejs","reactjs"]` and
`people = [["java"],["nodejs"],["nodejs","reactjs"]]`, the answer is `[0, 2]`:
person 0 brings java and person 2 brings both nodejs and reactjs, so two people
suffice where the greedy "most skills first" reading might have taken three.

**Constraints.** At most 16 skills and 60 people. Notice which of the two is
small: the mask indexes the **requirement**, not the people, so the table has
`2^16` entries and the 60 people are the transitions. Getting that the wrong way
round gives `2^60`. The answer is guaranteed to exist and to fit in 60 entries.

[leetcode.com/problems/smallest-sufficient-team](https://leetcode.com/problems/smallest-sufficient-team/)

In [24]:
def smallest_sufficient_team(req_skills, people):
    """The mask indexes the *requirement*, not the items.

    dp[covered] = the smallest team achieving exactly that coverage, so the
    table has 2^(number of skills) entries and the people are the transitions.
    """
    idx = {s: i for i, s in enumerate(req_skills)}
    m = len(req_skills)
    pmask = [sum(1 << idx[s] for s in p if s in idx) for p in people]
    full = (1 << m) - 1
    dp = {0: []}
    for i, pm in enumerate(pmask):
        if pm == 0:
            continue
        for covered, team in list(dp.items()):
            nc = covered | pm
            if nc == covered:
                continue
            if nc not in dp or len(dp[nc]) > len(team) + 1:
                dp[nc] = team + [i]
    return dp[full]

In [25]:
skills = ['java', 'nodejs', 'reactjs']
people = [['java'], ['nodejs'], ['nodejs', 'reactjs']]
team = smallest_sufficient_team(skills, people)
print(team, '->', [people[i] for i in team])

[0, 2] -> [['java'], ['nodejs', 'reactjs']]


## 13. LeetCode 526 — Beautiful Arrangement

**The task.** Count the permutations of `1..n` in which every element sits in a
position that divides it, or is divided by it. Writing the permutation as
`perm[1..n]`, position `i` must satisfy `perm[i] % i == 0` or `i % perm[i] == 0`.

**Input / output.** One integer `n`; return one integer, the number of such
permutations.

**Example.** For `n = 2` both orderings work: `[1,2]` because 1%1 and 2%2 are
zero, and `[2,1]` because 2%1 is zero and 1 divides position 2. The answer is 2.
For `n = 4` there are 8.

**Constraints.** `1 ≤ n ≤ 15`. Fifteen is far beyond `15! = 1.3 × 10^12`
permutations but comfortable for `2^15` masks. The lesson here is that the state
is *one* integer even though the problem is about orderings: the position being
filled is `popcount(mask) + 1`, so the mask already knows it and there is no
need to carry it separately.

[leetcode.com/problems/beautiful-arrangement](https://leetcode.com/problems/beautiful-arrangement/)

In [26]:
def count_arrangement(n):
    """The mask already knows the position: pos = popcount(mask) + 1.

    So the state is one integer even though the problem is about permutations.
    """
    full = (1 << n) - 1
    dp = [0] * (1 << n)
    dp[0] = 1
    for mask in range(1 << n):
        if dp[mask] == 0:
            continue
        pos = popcount(mask) + 1
        for v in range(1, n + 1):
            if has(mask, v - 1):
                continue
            if v % pos == 0 or pos % v == 0:
                dp[add(mask, v - 1)] += dp[mask]
    return dp[full]


def count_arrangement_brute(n):
    return sum(1 for p in itertools.permutations(range(1, n + 1))
               if all(v % (i + 1) == 0 or (i + 1) % v == 0 for i, v in enumerate(p)))

In [27]:
for n in (2, 3, 4, 8):
    print(f'n={n:2d} -> {count_arrangement(n):4d}  (brute force agrees:'
          f' {count_arrangement(n) == count_arrangement_brute(n)})')
print(f'n=15 -> {count_arrangement(15)}')

n= 2 ->    2  (brute force agrees: True)
n= 3 ->    3  (brute force agrees: True)
n= 4 ->    8  (brute force agrees: True)
n= 8 ->  132  (brute force agrees: True)
n=15 -> 24679


## 14. LeetCode 698 — Partition to K Equal Sum Subsets

**The task.** Decide whether an array can be split into exactly `k` groups whose
sums are all equal. Every element must land in exactly one group.

**Input / output.** An integer array `nums` and an integer `k`; return `True` or
`False`.

**Example.** `nums = [4,3,2,3,5,2,1]`, `k = 4` sums to 20, so each group must
total 5: `[5]`, `[1,4]`, `[2,3]`, `[2,3]`. The answer is `True`. `nums =
[1,2,3,4]` with `k = 3` sums to 10, which 3 does not divide, so it is `False`
before any searching happens.

**Constraints.** `nums.length ≤ 16`, values up to 10^4. Sixteen items is the
bitmask signal, and again the mask carries the extra state for free: once the
chosen set is fixed its sum is fixed, so `sum(mask) % target` tells you how full
the group in progress is — no bucket index, no running total. Two cheap
rejections come first: the total must divide by `k`, and no single element may
exceed the target.

[leetcode.com/problems/partition-to-k-equal-sum-subsets](https://leetcode.com/problems/partition-to-k-equal-sum-subsets/)

In [28]:
def can_partition_k_subsets(nums, k):
    """Again the mask carries the extra state for free.

    Once the chosen set is fixed, its sum is fixed, so sum(mask) % target tells
    you how full the bucket in progress is - no need to store a bucket index or
    a running total alongside the mask.
    """
    total = sum(nums)
    if total % k:
        return False
    target = total // k
    if max(nums) > target:
        return False
    n = len(nums)
    sums = [0] * (1 << n)
    for mask in range(1, 1 << n):
        low = mask & -mask                 # lowest set bit
        sums[mask] = sums[mask ^ low] + nums[low.bit_length() - 1]
    ok = [False] * (1 << n)
    ok[0] = True
    for mask in range(1 << n):
        if not ok[mask]:
            continue
        room = target - sums[mask] % target
        for i in range(n):
            if has(mask, i):
                continue
            if nums[i] <= room:
                ok[add(mask, i)] = True
    return ok[(1 << n) - 1]

In [29]:
for nums, k in (([4,3,2,3,5,2,1], 4), ([1,2,3,4], 3), ([2,2,2,2,3,4,5], 4)):
    print(f'{str(nums):<24} k={k} -> {can_partition_k_subsets(nums, k)}')

[4, 3, 2, 3, 5, 2, 1]    k=4 -> True
[1, 2, 3, 4]             k=3 -> False
[2, 2, 2, 2, 3, 4, 5]    k=4 -> False


## 15. Everything at once

If anything above drifts, this cell fails loudly.

In [30]:
assert best == cost == 689
assert wrong == mst_weight(DIST) == 502 < cost
assert ncost == 795
assert shortest_path_length([[1,2,3],[0],[0],[0]]) == 4
assert len(sup) == 16 and all(w in sup for w in words)
assert len(bad) == 18 and all(w in bad for w in words)
assert sorted(team) == [0, 2]
assert count_arrangement(8) == count_arrangement_brute(8) == 132
assert can_partition_k_subsets([4,3,2,3,5,2,1], 4)
assert not can_partition_k_subsets([1,2,3,4], 3)
assert subset_pair_count(10) == 3 ** 10
assert len(list(subsets_of(0b1011))) == len(list(subsets_of_buggy(0b1011))) + 1
print('all assertions passed')

all assertions passed
